# Insurance Claims Intelligence — 01: Create Lakehouse Data

Builds the `RF_Lakehouse` tables that everything else in this demo sits on:
the **InsuranceSM** semantic model, the **InsuranceOntology_ManualGen** ontology,
and the **InsuranceAgent** data agent.

**Run this first.** Nothing else works until these tables exist.

### Before you run
1. Create a lakehouse named `RF_Lakehouse` in your workspace.
2. Attach it to this notebook as the default lakehouse.
3. Run all cells (about a minute).

### What it creates
Eight Delta tables of synthetic property & casualty claims data spanning **Jan–Mar 2026**:
`offices`, `policyholders`, `insured_assets`, `adjusters`, `policies`, `claims`,
`claim_events`, `asset_inspections`.

The generator deliberately plants the edge cases the demo questions rely on —
NULL payouts, claims above coverage, uninspected assets, over-capacity adjusters,
and incomplete events. See the validation cell at the end.

## Configuration

In [ ]:
LAKEHOUSE_NAME = "RF_Lakehouse"   # must match the attached lakehouse
SEED           = 42               # fixed seed -> reproducible data

# Row counts. Small on purpose: the demo is about reasoning, not volume.
N_OFFICES       = 8
N_ADJUSTERS     = 24
N_POLICYHOLDERS = 180
N_ASSETS        = 220
N_POLICIES      = 240
N_CLAIMS        = 400
N_INSPECTIONS   = 150

## Schema

Column names and types must match exactly — the semantic model and ontology bind to them.

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    LongType, BooleanType, TimestampType,
)

SCHEMAS = {
    "offices": StructType([
        StructField("office_id", StringType()), StructField("name", StringType()),
        StructField("office_type", StringType()), StructField("address", StringType()),
        StructField("city", StringType()), StructField("state", StringType()),
        StructField("zip_code", LongType()), StructField("latitude", DoubleType()),
        StructField("longitude", DoubleType()), StructField("timezone", StringType()),
        StructField("phone", StringType()), StructField("adjuster_capacity", LongType()),
        StructField("has_siu_unit", BooleanType()),
    ]),
    "policyholders": StructType([
        StructField("policyholder_id", StringType()), StructField("policyholder_number", StringType()),
        StructField("full_name", StringType()), StructField("policyholder_type", StringType()),
        StructField("email", StringType()), StructField("phone", StringType()),
        StructField("address", StringType()), StructField("city", StringType()),
        StructField("state", StringType()), StructField("zip_code", LongType()),
        StructField("date_of_birth", TimestampType()), StructField("risk_score", LongType()),
    ]),
    "insured_assets": StructType([
        StructField("asset_id", StringType()), StructField("asset_number", StringType()),
        StructField("asset_type", StringType()), StructField("asset_description", StringType()),
        StructField("make", StringType()), StructField("model", StringType()),
        StructField("year", LongType()), StructField("vin", StringType()),
        StructField("address", StringType()), StructField("city", StringType()),
        StructField("state", StringType()), StructField("estimated_value", DoubleType()),
        StructField("condition", StringType()), StructField("last_inspection_date", TimestampType()),
    ]),
    "adjusters": StructType([
        StructField("adjuster_id", StringType()), StructField("employee_id", StringType()),
        StructField("first_name", StringType()), StructField("last_name", StringType()),
        StructField("email", StringType()), StructField("phone", StringType()),
        StructField("license_number", StringType()), StructField("license_state", StringType()),
        StructField("specializations", StringType()), StructField("hire_date", TimestampType()),
        StructField("status", StringType()), StructField("home_office_id", StringType()),
        StructField("supervisor_email", StringType()), StructField("max_active_claims", LongType()),
    ]),
    "policies": StructType([
        StructField("policy_id", StringType()), StructField("policy_number", StringType()),
        StructField("policyholder_id", StringType()), StructField("asset_id", StringType()),
        StructField("policy_type", StringType()), StructField("coverage_amount", DoubleType()),
        StructField("deductible", DoubleType()), StructField("premium_annual", DoubleType()),
        StructField("effective_date", TimestampType()), StructField("expiration_date", TimestampType()),
        StructField("status", StringType()), StructField("underwriter", StringType()),
    ]),
    "claims": StructType([
        StructField("claim_id", StringType()), StructField("claim_number", StringType()),
        StructField("policy_id", StringType()), StructField("policyholder_id", StringType()),
        StructField("asset_id", StringType()), StructField("adjuster_id", StringType()),
        StructField("office_id", StringType()), StructField("claim_type", StringType()),
        StructField("description", StringType()), StructField("incident_date", TimestampType()),
        StructField("filed_date", TimestampType()), StructField("status", StringType()),
        StructField("estimated_loss", DoubleType()), StructField("approved_amount", DoubleType()),
        StructField("priority", StringType()), StructField("incident_latitude", DoubleType()),
        StructField("incident_longitude", DoubleType()),
    ]),
    "claim_events": StructType([
        StructField("claim_event_id", StringType()), StructField("event_number", StringType()),
        StructField("claim_id", StringType()), StructField("adjuster_id", StringType()),
        StructField("event_type", StringType()), StructField("description", StringType()),
        StructField("status", StringType()), StructField("created_at", TimestampType()),
        StructField("completed_at", TimestampType()), StructField("notes", StringType()),
        StructField("cost_usd", DoubleType()),
    ]),
    "asset_inspections": StructType([
        StructField("inspection_id", StringType()), StructField("asset_id", StringType()),
        StructField("office_id", StringType()), StructField("inspection_type", StringType()),
        StructField("status", StringType()), StructField("scheduled_date", TimestampType()),
        StructField("completed_date", TimestampType()), StructField("appraised_value", DoubleType()),
        StructField("inspector_notes", StringType()), StructField("condition_rating", StringType()),
    ]),
}

print(f"{len(SCHEMAS)} table schemas defined")

## Reference data

In [ ]:
import random
from datetime import datetime, timedelta

rng = random.Random(SEED)

CLAIM_TYPES = ["fire", "property_damage", "liability", "auto_collision",
               "weather", "auto_theft", "water_damage"]
CLAIM_STATUSES = ["filed", "under_review", "investigation", "approved", "paid", "denied"]
PRIORITIES = ["low", "medium", "high", "urgent"]
EVENT_TYPES = ["inspection", "appraisal", "interview", "documentation",
               "siu_review", "settlement_call", "repair_estimate"]

OFFICE_SEED = [
    ("Dallas Claims Center",    "regional",   "Dallas",      "TX", 75201, 32.7767,  -96.7970, "America/Chicago"),
    ("Houston Claims Center",   "regional",   "Houston",     "TX", 77002, 29.7604,  -95.3698, "America/Chicago"),
    ("Phoenix Field Office",    "field",      "Phoenix",     "AZ", 85004, 33.4484, -112.0740, "America/Phoenix"),
    ("Denver Field Office",     "field",      "Denver",      "CO", 80202, 39.7392, -104.9903, "America/Denver"),
    ("Atlanta Claims Center",   "regional",   "Atlanta",     "GA", 30303, 33.7490,  -84.3880, "America/New_York"),
    ("Tampa Field Office",      "field",      "Tampa",       "FL", 33602, 27.9506,  -82.4572, "America/New_York"),
    ("Kansas City Satellite",   "satellite",  "Kansas City", "MO", 64106, 39.0997,  -94.5786, "America/Chicago"),
    ("Nashville Satellite",     "satellite",  "Nashville",   "TN", 37203, 36.1627,  -86.7816, "America/Chicago"),
]

FIRST = ["James","Maria","Robert","Linda","Michael","Patricia","David","Jennifer","William","Elizabeth",
         "Carlos","Aisha","Daniel","Priya","Kevin","Sofia","Brian","Nina","Marcus","Grace",
         "Andre","Yuki","Omar","Rachel"]
LAST = ["Alvarez","Bennett","Chen","Douglas","Ellis","Foster","Gupta","Hayes","Ibrahim","Jensen",
        "Kowalski","Lindqvist","Mbeki","Novak","Oyelaran","Park","Quinn","Rivera","Sandoval","Tanaka",
        "Ustinov","Vargas","Whitfield","Zhao"]

VEHICLE_MAKES = {"Toyota": ["Camry","RAV4","Tacoma"], "Ford": ["F-150","Explorer","Escape"],
                 "Honda": ["Civic","CR-V","Accord"], "Chevrolet": ["Silverado","Equinox","Malibu"]}
BUILDING_MAKES = {"Residential": ["Single Family","Townhome","Condo"],
                  "Commercial": ["Retail Strip","Warehouse","Office Suite"]}

STREETS = ["Oak","Maple","Cedar","Pine","Elm","Birch","Walnut","Aspen","Juniper","Willow"]

WINDOW_START = datetime(2026, 1, 1)
WINDOW_END   = datetime(2026, 3, 31)

def rand_dt(start=WINDOW_START, end=WINDOW_END):
    delta = int((end - start).total_seconds())
    return start + timedelta(seconds=rng.randint(0, delta))

def money(lo, hi, step=50):
    return float(round(rng.uniform(lo, hi) / step) * step)

print("reference data ready")

## Generate rows

In [ ]:
# ---- offices -------------------------------------------------------------
offices = []
for i, (name, otype, city, state, zc, lat, lon, tz) in enumerate(OFFICE_SEED[:N_OFFICES], start=1):
    offices.append((
        f"OFF-{i:03d}", name, otype,
        f"{rng.randint(100, 9999)} {rng.choice(STREETS)} St", city, state, int(zc),
        float(lat), float(lon), tz,
        f"({rng.randint(200,989)}) 555-{rng.randint(1000,9999)}",
        int(rng.choice([6, 8, 10, 12])),
        otype == "regional",
    ))
office_ids = [o[0] for o in offices]

# ---- adjusters -----------------------------------------------------------
adjusters = []
for i in range(1, N_ADJUSTERS + 1):
    fn, ln = FIRST[(i - 1) % len(FIRST)], LAST[(i - 1) % len(LAST)]
    home = rng.choice(office_ids)
    state = next(o[5] for o in offices if o[0] == home)
    adjusters.append((
        f"ADJ-{i:03d}", f"E{10000 + i}", fn, ln,
        f"{fn.lower()}.{ln.lower()}@example-insurance.com",
        f"({rng.randint(200,989)}) 555-{rng.randint(1000,9999)}",
        f"LIC-{state}-{rng.randint(100000, 999999)}", state,
        rng.choice(["auto", "property", "auto;property", "liability", "catastrophe"]),
        datetime(rng.randint(2012, 2025), rng.randint(1, 12), rng.randint(1, 28)),
        "active" if i % 13 else "inactive", home,
        f"supervisor{rng.randint(1,4)}@example-insurance.com",
        int(rng.choice([8, 10, 12, 15])),
    ))
adjuster_ids = [a[0] for a in adjusters]

# ---- policyholders -------------------------------------------------------
policyholders = []
for i in range(1, N_POLICYHOLDERS + 1):
    is_org = i % 7 == 0
    fn, ln = rng.choice(FIRST), rng.choice(LAST)
    name = f"{ln} {rng.choice(['Holdings','Logistics','Properties','Retail Group'])}" if is_org else f"{fn} {ln}"
    off = rng.choice(offices)
    policyholders.append((
        f"PH-{i:04d}", f"PH{200000 + i}", name,
        "organization" if is_org else "individual",
        f"{fn.lower()}.{ln.lower()}{i}@example.com",
        f"({rng.randint(200,989)}) 555-{rng.randint(1000,9999)}",
        f"{rng.randint(100, 9999)} {rng.choice(STREETS)} Ave",
        off[3], off[5], int(off[6]),
        None if is_org else datetime(rng.randint(1955, 2001), rng.randint(1, 12), rng.randint(1, 28)),
        int(rng.randint(300, 850)),
    ))
policyholder_ids = [p[0] for p in policyholders]

print(f"offices={len(offices)} adjusters={len(adjusters)} policyholders={len(policyholders)}")

In [ ]:
# ---- insured_assets ------------------------------------------------------
# EDGE CASE: ~18% get last_inspection_date = NULL ("never inspected").
insured_assets = []
for i in range(1, N_ASSETS + 1):
    is_vehicle = i % 2 == 0
    off = rng.choice(offices)
    if is_vehicle:
        make = rng.choice(list(VEHICLE_MAKES)); model = rng.choice(VEHICLE_MAKES[make])
        atype, year = "vehicle", int(rng.randint(2014, 2026))
        vin = "".join(rng.choice("ABCDEFGHJKLMNPRSTUVWXYZ0123456789") for _ in range(17))
        value, desc = money(8000, 68000, 100), f"{year} {make} {model}"
    else:
        make = rng.choice(list(BUILDING_MAKES)); model = rng.choice(BUILDING_MAKES[make])
        atype, year = "building", int(rng.randint(1960, 2024))
        vin, value, desc = None, money(120000, 1400000, 1000), f"{model} built {year}"
    insured_assets.append((
        f"AST-{i:04d}", f"AS{300000 + i}", atype, desc, make, model, year, vin,
        f"{rng.randint(100, 9999)} {rng.choice(STREETS)} Rd", off[3], off[5],
        value, rng.choice(["excellent", "good", "fair", "poor"]),
        None if rng.random() < 0.18 else rand_dt(datetime(2024, 1, 1), datetime(2026, 3, 1)),
    ))
asset_ids = [a[0] for a in insured_assets]
asset_value = {a[0]: a[11] for a in insured_assets}

# ---- policies ------------------------------------------------------------
policies = []
for i in range(1, N_POLICIES + 1):
    aid = asset_ids[(i - 1) % len(asset_ids)]
    eff = datetime(2025, rng.randint(1, 12), rng.randint(1, 28))
    coverage = float(round(asset_value[aid] * rng.uniform(0.75, 1.15) / 500) * 500)
    policies.append((
        f"POL-{i:04d}", f"PN{400000 + i}", rng.choice(policyholder_ids), aid,
        rng.choice(["auto", "homeowners", "commercial_property", "umbrella"]),
        coverage, money(500, 5000, 250), money(600, 9000, 50),
        eff, eff + timedelta(days=365),
        rng.choice(["active", "active", "active", "lapsed", "cancelled"]),
        f"{rng.choice(FIRST)} {rng.choice(LAST)}",
    ))
policy_ids = [p[0] for p in policies]
policy_coverage = {p[0]: p[5] for p in policies}
policy_asset = {p[0]: p[3] for p in policies}

print(f"assets={len(insured_assets)} policies={len(policies)}")
print(f"  never-inspected assets: {sum(1 for a in insured_assets if a[13] is None)}")

In [ ]:
# ---- claims --------------------------------------------------------------
# EDGE CASES planted here:
#   * approved_amount IS NULL unless status is approved/paid  -> "unpaid claims"
#   * ~8% have estimated_loss > policy coverage_amount        -> "exceeds coverage"
#   * a few adjusters are deliberately overloaded             -> "over capacity"
claims = []
overloaded = adjuster_ids[:3]   # these three will blow past max_active_claims

for i in range(1, N_CLAIMS + 1):
    pol = rng.choice(policy_ids)
    aid = policy_asset[pol]
    off = rng.choice(offices)
    status = rng.choice(CLAIM_STATUSES)

    # bias the first ~45 claims onto the overloaded adjusters, and keep them open
    if i <= 45:
        adj = overloaded[i % len(overloaded)]
        status = rng.choice(["filed", "under_review", "investigation", "approved"])
    else:
        adj = rng.choice(adjuster_ids)

    coverage = policy_coverage[pol]
    if rng.random() < 0.08:
        est = float(round(coverage * rng.uniform(1.05, 1.6) / 50) * 50)   # over coverage
    else:
        est = float(round(coverage * rng.uniform(0.05, 0.85) / 50) * 50)
    est = max(est, 500.0)

    # approved_amount is NULL unless the claim actually reached a payout state
    if status in ("approved", "paid"):
        approved = float(round(min(est, coverage) * rng.uniform(0.55, 0.98) / 50) * 50)
    else:
        approved = None

    incident = rand_dt()
    filed = incident + timedelta(days=rng.randint(0, 9))
    if filed > WINDOW_END:
        filed = WINDOW_END
    ctype = rng.choice(CLAIM_TYPES)

    claims.append((
        f"CLM-{i:05d}", f"CN{500000 + i}", pol, rng.choice(policyholder_ids), aid, adj, off[0],
        ctype, f"{ctype.replace('_', ' ').title()} reported at insured location",
        incident, filed, status, est, approved,
        rng.choice(PRIORITIES),
        round(off[7] + rng.uniform(-0.4, 0.4), 6),
        round(off[8] + rng.uniform(-0.4, 0.4), 6),
    ))
claim_ids = [c[0] for c in claims]

print(f"claims={len(claims)}")
print(f"  NULL approved_amount : {sum(1 for c in claims if c[13] is None)}")
print(f"  over policy coverage : {sum(1 for c in claims if c[12] > policy_coverage[c[2]])}")

In [ ]:
# ---- claim_events --------------------------------------------------------
# EDGE CASE: ~30% have completed_at = NULL -> "incomplete events".
# Note cost_usd is handling cost, NOT payout. Never add it to approved_amount.
claim_events, n = [], 0
for cid in claim_ids:
    for _ in range(rng.randint(0, 4)):
        n += 1
        created = rand_dt()
        incomplete = rng.random() < 0.30
        etype = rng.choice(EVENT_TYPES)
        claim_events.append((
            f"EVT-{n:06d}", f"EN{600000 + n}", cid, rng.choice(adjuster_ids), etype,
            f"{etype.replace('_', ' ').title()} on claim",
            "in_progress" if incomplete else "completed",
            created,
            None if incomplete else created + timedelta(hours=rng.randint(2, 120)),
            rng.choice(["Awaiting documentation", "Photos received", "Follow-up scheduled", None]),
            money(75, 2400, 25),
        ))

# ---- asset_inspections ---------------------------------------------------
asset_inspections = []
for i in range(1, N_INSPECTIONS + 1):
    sched = rand_dt()
    pending = rng.random() < 0.25
    aid = rng.choice(asset_ids)
    asset_inspections.append((
        f"INS-{i:05d}", aid, rng.choice(office_ids),
        rng.choice(["initial", "periodic", "post_claim", "reinspection"]),
        "scheduled" if pending else "completed", sched,
        None if pending else sched + timedelta(days=rng.randint(1, 21)),
        None if pending else float(round(asset_value[aid] * rng.uniform(0.85, 1.1) / 100) * 100),
        None if pending else rng.choice(["No damage observed", "Minor wear noted", "Repairs recommended"]),
        None if pending else rng.choice(["A", "B", "C", "D"]),
    ))

print(f"claim_events={len(claim_events)}  (incomplete: {sum(1 for e in claim_events if e[8] is None)})")
print(f"asset_inspections={len(asset_inspections)}")

## Write Delta tables

In [ ]:
DATA = {
    "offices": offices,
    "policyholders": policyholders,
    "insured_assets": insured_assets,
    "adjusters": adjusters,
    "policies": policies,
    "claims": claims,
    "claim_events": claim_events,
    "asset_inspections": asset_inspections,
}

for name, rows in DATA.items():
    df = spark.createDataFrame(rows, schema=SCHEMAS[name])
    df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(name)
    print(f"  wrote {name:<20} {df.count():>6} rows")

print("\nAll tables written.")

## Validate

Every check below backs a question the data agent is expected to answer.
All of them should return a non-zero count — if any is zero, the matching
demo question will come back empty.

In [ ]:
checks = {
    "claims total":
        "SELECT COUNT(*) FROM claims",
    "open claims (not paid/denied)":
        "SELECT COUNT(*) FROM claims WHERE status NOT IN ('paid','denied')",
    "claims with NULL approved_amount":
        "SELECT COUNT(*) FROM claims WHERE approved_amount IS NULL",
    "claims exceeding policy coverage":
        "SELECT COUNT(*) FROM claims c JOIN policies p ON c.policy_id = p.policy_id "
        "WHERE c.estimated_loss > p.coverage_amount",
    "assets never inspected":
        "SELECT COUNT(*) FROM insured_assets WHERE last_inspection_date IS NULL",
    "incomplete claim events":
        "SELECT COUNT(*) FROM claim_events WHERE completed_at IS NULL",
    "adjusters over capacity":
        "SELECT COUNT(*) FROM (SELECT c.adjuster_id, COUNT(*) AS open_ct "
        "FROM claims c WHERE c.status NOT IN ('paid','denied') GROUP BY c.adjuster_id) w "
        "JOIN adjusters a ON a.adjuster_id = w.adjuster_id WHERE w.open_ct > a.max_active_claims",
}

print(f"{'check':<38} {'count':>7}")
print('-' * 46)
failed = []
for label, sql in checks.items():
    v = spark.sql(sql).collect()[0][0]
    print(f"{label:<38} {v:>7}")
    if v == 0:
        failed.append(label)

print()
print("All demo edge cases present." if not failed else f"EMPTY: {failed}")

---

### Next

1. `02_create_ontology.ipynb` — builds the ontology with bound data.
2. Create the **InsuranceSM** semantic model over these tables (see `semantic-model/`).
3. Configure the data agent (see `data-agent/README.md`).